# Overview of Expression Prediction Evaluation

The expression prediction eval is the primary benchmark for BioJEPA's predictive capability. Given a cell's control state and a perturbation, the model predicts the resulting gene expression profile, and we evaluate how well those predictions match reality.

This notebook picks up where the [Eval Decoders](explainer_eval_decoders.ipynb) notebook left off. The decoder has already converted latent representations into expression predictions. Here we take those predictions and compute every metric in `_compute_expression_prediction`.

We won't use any neural network code here. Everything operates on numpy arrays: predicted expression, real expression, and control baselines. The goal is to build intuition for what each metric measures and, just as importantly, what it misses.

*Note: production also computes `sample_level` metrics during the inference loop (before perturbation averaging). Those are not covered here since they aren't part of `_compute_expression_prediction`.*

In [1]:
import numpy as np
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import r2_score

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


In [2]:
SEED = 1337
np.random.seed(SEED)

num_perts = 4
num_genes = 8
TOP_K = 3

## The Four Perturbation Archetypes

Rather than generating random data, we'll hand-craft four perturbations that each expose a different failure mode. This makes the metrics' behavior visible in a way that random numbers can't.

| Pert | Profile | What it teaches |
|------|---------|-----------------|
| 0 | Strong effect, well-predicted | The "happy path": high R$^2$, high Pearson, low MSE |
| 1 | Weak effect, tiny deltas | Baseline is competitive when perturbation barely changes expression |
| 2 | Wrong direction on several genes | Pearson on deltas collapses, centroid gets matched to the wrong perturbation |
| 3 | Right direction, ~3x overestimated magnitude | Perfect Pearson on deltas (1.0) but terrible R$^2$, because Pearson is scale-invariant |

*In production, we use `TOP_K = 50` (the GEARS benchmark standard). With only 8 genes, that would select everything, so we use `TOP_K = 3` here.*

In [3]:
control = np.array([2.0, 3.5, 1.0, 4.0, 2.5, 3.0, 1.5, 5.0])

real_delta = np.array([
    [ 1.5 , -2.0 ,  0.8 , -1.2 ,  0.3 ,  2.5 , -0.5 ,  1.0 ],
    [ 0.05, -0.03,  0.02,  0.04, -0.01,  0.03, -0.02,  0.01],
    [ 1.0 , -1.5 ,  0.5 ,  0.8 , -0.3 ,  1.2 , -0.7 ,  0.4 ],
    [ 0.5 , -1.0 ,  0.3 ,  0.8 , -0.2 ,  1.5 , -0.4 ,  0.6 ],
])

pred_delta = np.array([
    [ 1.4 , -1.8 ,  0.9 , -1.1 ,  0.2 ,  2.3 , -0.6 ,  0.9 ],
    [ 0.08, -0.05,  0.01,  0.06, -0.03,  0.02, -0.04,  0.03],
    [-0.5 ,  0.8 ,  0.3 , -0.6 ,  0.2 , -0.9 ,  0.4 ,  0.5 ],
    [ 1.5 , -3.0 ,  0.9 ,  2.4 , -0.6 ,  4.5 , -1.2 ,  1.8 ],
])

We derive absolute expression by adding deltas to the control baseline. In production, each perturbation can have a different mean control (from different batches or cell types). We use a shared control here for simplicity.

In [4]:
real_abs = control + real_delta
pred_abs = control + pred_delta

real_abs

array([[3.5 , 1.5 , 1.8 , 2.8 , 2.8 , 5.5 , 1.  , 6.  ],
       [2.05, 3.47, 1.02, 4.04, 2.49, 3.03, 1.48, 5.01],
       [3.  , 2.  , 1.5 , 4.8 , 2.2 , 4.2 , 0.8 , 5.4 ],
       [2.5 , 2.5 , 1.3 , 4.8 , 2.3 , 4.5 , 1.1 , 5.6 ]])

## Per-Perturbation Metrics

We'll compute six metrics for each perturbation individually, then aggregate across perturbations at the end.

### R$^2$ on All Genes

$R^2$ (coefficient of determination) measures how much variance in the real expression our predictions explain:

$$R^2 = 1 - \frac{\sum_i (y_i - \hat{y}_i)^2}{\sum_i (y_i - \bar{y})^2}$$

where $y$ is real absolute expression and $\hat{y}$ is predicted absolute expression. A perfect prediction gives $R^2 = 1$. Predictions worse than predicting the mean give $R^2 < 0$.

You'll see that pert 0 scores near 1.0 (good predictions), while pert 2 and pert 3 score much lower.

In [5]:
per_pert_r2_all = []
for i in range(num_perts):
    if np.std(real_abs[i]) > 1e-9:
        per_pert_r2_all.append(r2_score(real_abs[i], pred_abs[i]))
per_pert_r2_all = np.array(per_pert_r2_all)
per_pert_r2_all

array([0.99395738, 0.99975177, 0.1955657 , 0.02716425])

### Top-K DEG Selection

Not all genes respond equally to a perturbation. Most barely change, while a handful of differentially expressed genes (DEGs) shift dramatically. Evaluating only on the top-K DEGs focuses the metric on the genes that matter most biologically.

We select the top-K genes by sorting on $|\text{real\_delta}|$, the magnitude of the actual expression change. Let's see this on pert 0.

In [6]:
i = 0
magnitudes = np.abs(real_delta[i])
sorted_idx = np.argsort(magnitudes)

for idx in sorted_idx:
    marker = ' <-- top-K' if idx in sorted_idx[-TOP_K:] else ''
    print(f'  gene {idx}: |delta| = {magnitudes[idx]:.1f}{marker}')

  gene 4: |delta| = 0.3
  gene 6: |delta| = 0.5
  gene 2: |delta| = 0.8
  gene 7: |delta| = 1.0
  gene 3: |delta| = 1.2
  gene 0: |delta| = 1.5 <-- top-K
  gene 1: |delta| = 2.0 <-- top-K
  gene 5: |delta| = 2.5 <-- top-K


### R$^2$ on Top-K DEGs

Now we compute $R^2$ on just those top-K genes. This is where the first key teaching moment appears.

You'll see that pert 3 has a *negative* $R^2$ on the top-K DEGs despite having a positive (if low) $R^2$ on all genes. The reason: pert 3's predictions overshoot by 3x. On the majority of genes where the real delta is small, the control baseline dominates and the 3x error is modest in absolute terms. But on the top-K genes, where the real deltas are largest, the 3x overshoot produces enormous residuals that blow up the numerator of $R^2$.

In [7]:
per_pert_r2_topk = []
for i in range(num_perts):
    top_k_idx = np.argsort(np.abs(real_delta[i]))[-TOP_K:]
    per_pert_r2_topk.append(r2_score(real_abs[i, top_k_idx], pred_abs[i, top_k_idx]))
per_pert_r2_topk = np.array(per_pert_r2_topk)
per_pert_r2_topk

array([ 0.98875   ,  0.999293  , -3.92445055, -3.97654584])

### MSE on Deltas

Mean squared error on deltas is the most straightforward metric: how far off are our predicted changes from the real changes, on average?

$$\text{MSE}_i = \frac{1}{G} \sum_{g=1}^{G} (\hat{\delta}_{ig} - \delta_{ig})^2$$

Pert 1 will have the smallest MSE (tiny deltas, tiny errors). Pert 3 will have the largest (3x overshoot on every gene).

In [8]:
per_pert_mse = np.mean((pred_delta - real_delta)**2, axis=1)
per_pert_mse

array([1.7500e-02, 3.8750e-04, 1.9275e+00, 2.3950e+00])

### Pearson Correlation on Absolute Expression

Pearson correlation on absolute expression measures whether the predicted and real expression profiles have the same shape, regardless of scale:

$$r = \frac{\sum_i (x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum_i (x_i - \bar{x})^2} \sqrt{\sum_i (y_i - \bar{y})^2}}$$

You'll see that all four perturbations score high on this metric. This is because the control baseline dominates the absolute expression profile. Even pert 2, which gets the directions wrong, still has the control pattern baked in (the 2.0, 3.5, 1.0, ... baseline drives the correlation).

Production uses a NaN guard: `pearsonr` can return NaN for near-constant inputs even when `std > 1e-9`. We check both conditions.

In [9]:
per_pert_pearson_abs = []
for i in range(num_perts):
    if np.std(pred_abs[i]) > 1e-9 and np.std(real_abs[i]) > 1e-9:
        r, _ = pearsonr(pred_abs[i], real_abs[i])
        per_pert_pearson_abs.append(0.0 if np.isnan(r) else float(r))
    else:
        per_pert_pearson_abs.append(0.0)
per_pert_pearson_abs = np.array(per_pert_pearson_abs)
per_pert_pearson_abs

array([0.99793172, 0.99989898, 0.55965689, 0.90963928])

### Pearson Correlation on Deltas

Pearson on deltas strips away the control baseline and measures whether the model predicts the right *pattern* of changes. This is a much harder test.

Here's the second key teaching moment. Pert 3 predicts exactly `3 * real_delta` for every gene. Pearson correlation is scale-invariant: multiplying every value by a constant doesn't change the correlation. So pert 3 will have a *perfect* Pearson on deltas (1.0) despite having a deeply negative R$^2$ on the top-K DEGs.

This is exactly why we need both metrics. Pearson tells you "the model knows which genes go up and which go down." R$^2$ tells you "the model gets the magnitudes right." Pert 3 has the first but not the second.

In [10]:
per_pert_pearson_delta = []
for i in range(num_perts):
    if np.std(pred_delta[i]) > 1e-9 and np.std(real_delta[i]) > 1e-9:
        r, _ = pearsonr(pred_delta[i], real_delta[i])
        per_pert_pearson_delta.append(0.0 if np.isnan(r) else float(r))
    else:
        per_pert_pearson_delta.append(0.0)
per_pert_pearson_delta = np.array(per_pert_pearson_delta)
per_pert_pearson_delta

array([ 0.99794143,  0.95997532, -0.82735815,  1.        ])

### Pearson on Top-K DEGs

Finally, Pearson correlation restricted to the top-K differentially expressed genes. This combines the delta-only perspective of Pearson with the DEG focus of top-K selection.

In [11]:
per_pert_pearson_topk = []
for i in range(num_perts):
    top_k_idx = np.argsort(np.abs(real_delta[i]))[-TOP_K:]
    pd_top, rd_top = pred_delta[i, top_k_idx], real_delta[i, top_k_idx]
    if np.std(pd_top) > 1e-9 and np.std(rd_top) > 1e-9:
        r, _ = pearsonr(pd_top, rd_top)
        per_pert_pearson_topk.append(0.0 if np.isnan(r) else float(r))
    else:
        per_pert_pearson_topk.append(0.0)
per_pert_pearson_topk = np.array(per_pert_pearson_topk)
per_pert_pearson_topk

array([ 0.99999598,  0.98198051, -0.98715676,  1.        ])

## Cross-Perturbation Metrics

The metrics above evaluate each perturbation in isolation. The next set looks across all perturbations simultaneously: can the model distinguish one perturbation's effect from another?

### Centroid Accuracy

For each perturbation, we ask: is the predicted delta vector closest (in squared Euclidean distance) to the corresponding real delta vector? If every perturbation's prediction is nearest to its own ground truth, centroid accuracy is 1.0.

We compute the full pairwise distance matrix using the expanded form:

$$\|a - b\|^2 = \|a\|^2 + \|b\|^2 - 2 \, a \cdot b$$

This avoids an explicit loop over perturbation pairs.

In [12]:
pred_sq = np.sum(pred_delta**2, axis=1)
real_sq = np.sum(real_delta**2, axis=1)
dist_matrix = pred_sq[:, None] + real_sq[None, :] - 2.0 * pred_delta @ real_delta.T
dist_matrix

array([[1.40000e-01, 1.33549e+01, 5.74000e+00, 6.35000e+00],
       [1.54424e+01, 3.10000e-03, 5.77440e+00, 4.38440e+00],
       [2.50800e+01, 2.80490e+00, 1.54200e+01, 1.27700e+01],
       [1.99100e+01, 4.21929e+01, 1.84100e+01, 1.91600e+01]])

Each row is a predicted perturbation, each column is a real perturbation. We take the `argmin` of each row to find the nearest real perturbation. Pert 2's prediction has flipped directions, so its predicted delta vector points away from its own real delta and closer to another perturbation's (likely pert 1, whose near-zero real delta is closer to anything than pert 2's real delta is to pert 2's flipped prediction).

*In production, centroid accuracy is 0.034 across 1,085 perturbations (37x better than random, but still quite low in absolute terms).*

In [13]:
matches = np.argmin(dist_matrix, axis=1)
centroid_acc = float(np.mean(matches == np.arange(num_perts)))
matches, centroid_acc

(array([0, 1, 1, 2]), 0.5)

### Vs Baseline Beat Rate

This metric asks a pointed question: does the model's prediction correlate better with real expression than the control state does? For each perturbation, we compute Pearson correlation of both `pred_abs` and `control` against `real_abs`, then count how often the model wins.

When a perturbation barely changes expression (like pert 1), the control is an excellent predictor of the perturbed state. The model has to do better than just returning the input unchanged.

*In v0.6 production, the beat rate is 0.009. Control is very competitive for single-gene perturbations because most of the 10,000-gene expression profile is unaffected.*

In [14]:
n_beat, n_eval = 0, 0
for i in range(num_perts):
    if np.std(real_abs[i]) < 1e-9:
        continue
    r_model, _ = pearsonr(pred_abs[i], real_abs[i])
    r_baseline, _ = pearsonr(control, real_abs[i])
    r_model = 0.0 if np.isnan(r_model) else r_model
    r_baseline = 0.0 if np.isnan(r_baseline) else r_baseline
    n_eval += 1
    if r_model > r_baseline:
        n_beat += 1
beat_rate = n_beat / n_eval if n_eval > 0 else 0.0
n_beat, n_eval, beat_rate

(3, 4, 0.75)

### Severity Correlation

Severity measures the overall magnitude of a perturbation's effect as the L2 norm of its delta vector:

$$\text{severity}_i = \|\delta_i\|_2 = \sqrt{\sum_{g=1}^{G} \delta_{ig}^2}$$

We compute this for both predicted and real deltas, then check whether the ranking of perturbations by severity is preserved. Pearson captures the linear relationship, Spearman captures the rank ordering.

Pert 3's 3x magnitude inflation will make it appear as the most severe perturbation in predictions, even though it's only the third most severe in reality. This creates ranking disagreement.

In [15]:
pred_severity = np.linalg.norm(pred_delta, axis=1)
real_severity = np.linalg.norm(real_delta, axis=1)

severity_table = np.column_stack([real_severity, pred_severity])
severity_table

array([[3.98998747, 3.70405184],
       [0.08306624, 0.12806248],
       [2.51396102, 1.61245155],
       [2.18860686, 6.56582059]])

In [16]:
r_sev_p, _ = pearsonr(pred_severity, real_severity)
r_sev_s, _ = spearmanr(pred_severity, real_severity)
severity_pearson = 0.0 if np.isnan(r_sev_p) else float(r_sev_p)
severity_spearman = 0.0 if np.isnan(r_sev_s) else float(r_sev_s)
severity_pearson, severity_spearman

(0.5087442026774452, 0.39999999999999997)

### Error by Magnitude

The final cross-perturbation metric bins all gene-level predictions by the magnitude of their real delta, then computes mean absolute error (MAE) within each bin. This reveals whether the model's errors scale with the size of the effect.

We flatten across all perturbations to get a single gene-level view, then bin by $|\text{real\_delta}|$ using the same bin edges as production.

In [17]:
all_pred = pred_delta.flatten()
all_real = real_delta.flatten()
all_errors = all_pred - all_real
all_magnitudes = np.abs(all_real)

magnitude_bins = [0, 0.25, 0.5, 1.0, 1.5, 2.0, np.inf]
bin_labels = ['0-0.25', '0.25-0.5', '0.5-1.0', '1.0-1.5', '1.5-2.0', '2.0+']

error_by_magnitude = {}
for j in range(len(magnitude_bins) - 1):
    mask = (all_magnitudes >= magnitude_bins[j]) & (all_magnitudes < magnitude_bins[j + 1])
    if mask.sum() > 0:
        error_by_magnitude[bin_labels[j]] = {'mae': float(np.mean(np.abs(all_errors[mask]))), 'count': int(mask.sum())}
error_by_magnitude

{'0-0.25': {'mae': 0.0611111111111111, 'count': 9},
 '0.25-0.5': {'mae': 0.42000000000000004, 'count': 5},
 '0.5-1.0': {'mae': 0.8375, 'count': 8},
 '1.0-1.5': {'mae': 1.16, 'count': 5},
 '1.5-2.0': {'mae': 1.8, 'count': 3},
 '2.0+': {'mae': 0.20000000000000007, 'count': 2}}

## Aggregation

Production reports both mean and median for each per-perturbation metric. The median is more robust to outliers, which matters here: pert 3's extreme R$^2$ values will pull the mean away from the median.

In [18]:
summary = {}
for name, values in [
    ('r2_all_genes', per_pert_r2_all), ('r2_topk_degs', per_pert_r2_topk),
    ('mse', per_pert_mse), ('pearson_all_genes', per_pert_pearson_abs),
    ('pearson_delta_all_genes', per_pert_pearson_delta), ('pearson_topk_degs', per_pert_pearson_topk),
]:
    summary[name] = {'mean': float(values.mean()), 'median': float(np.median(values))}
summary

{'r2_all_genes': {'mean': 0.5541097753987336, 'median': 0.5947615386570706},
 'r2_topk_degs': {'mean': -1.4782383477402588, 'median': -1.4678502747252742},
 'mse': {'mean': 1.085096875, 'median': 0.9725},
 'pearson_all_genes': {'mean': 0.8667817182485695,
  'median': 0.9537854996324002},
 'pearson_delta_all_genes': {'mean': 0.5326396482676916,
  'median': 0.9789583737605277},
 'pearson_topk_degs': {'mean': 0.49870493095149127,
  'median': 0.9909882440481508}}

In [19]:
results = {
    'perturbation_level': summary,
    'centroid_accuracy': centroid_acc,
    'vs_baseline': {'beat_rate': beat_rate, 'n_evaluated': n_eval},
    'severity': {'pearson_r': severity_pearson, 'spearman_r': severity_spearman},
    'error_by_magnitude': error_by_magnitude,
}
results

{'perturbation_level': {'r2_all_genes': {'mean': 0.5541097753987336,
   'median': 0.5947615386570706},
  'r2_topk_degs': {'mean': -1.4782383477402588, 'median': -1.4678502747252742},
  'mse': {'mean': 1.085096875, 'median': 0.9725},
  'pearson_all_genes': {'mean': 0.8667817182485695,
   'median': 0.9537854996324002},
  'pearson_delta_all_genes': {'mean': 0.5326396482676916,
   'median': 0.9789583737605277},
  'pearson_topk_degs': {'mean': 0.49870493095149127,
   'median': 0.9909882440481508}},
 'centroid_accuracy': 0.5,
 'vs_baseline': {'beat_rate': 0.75, 'n_evaluated': 4},
 'severity': {'pearson_r': 0.5087442026774452,
  'spearman_r': 0.39999999999999997},
 'error_by_magnitude': {'0-0.25': {'mae': 0.0611111111111111, 'count': 9},
  '0.25-0.5': {'mae': 0.42000000000000004, 'count': 5},
  '0.5-1.0': {'mae': 0.8375, 'count': 8},
  '1.0-1.5': {'mae': 1.16, 'count': 5},
  '1.5-2.0': {'mae': 1.8, 'count': 3},
  '2.0+': {'mae': 0.20000000000000007, 'count': 2}}}

## Expression Prediction Evaluation

No single metric tells the full story:

- **Pert 3** has perfect Pearson on deltas (1.0) but deeply negative top-K R$^2$. Pearson is scale-invariant, R$^2$ is not. The model knows *which* genes change, but not *by how much*.
- **Pert 2** has decent Pearson on absolute values (~0.6) but collapsed Pearson on deltas (~-0.8). The control baseline masks directional errors when you look at absolute expression. It's also the only perturbation where the baseline beats the model outright.
- **Pert 1** has excellent R$^2$ (0.9998) and negligible MSE, yet barely edges past the control baseline. When perturbation effects are tiny, the margin of victory over doing nothing is paper-thin.

This is why `_compute_expression_prediction` reports so many metrics, each catching a different failure mode. In production, the results dictionary also includes `sample_level` (per-sample MSE and correlation from the inference loop), `by_dataset` (all metrics broken down per dataset), and `gears_benchmark` (official GEARS comparison on Replogle K562 and Adamson).